In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px

from utils import csv_downloader

In [2]:
df_cohort = csv_downloader(URL = "https://raw.githubusercontent.com/hovhannisyan91/data_analytics_with_python/refs/heads/main/data/cohort/cohort_analysis.csv",
                           name = 'cohort',
                           path = '../data/raw')

cohort saved ../data/raw | shape : (12000, 12)


In [3]:
# Reading the Data
df_cohort = pd.read_csv('../data/raw/cohort.csv')
df_cohort.head()

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type
0,1,2024-07-01,2025-02-01,Male,Married,31,Medium,Germany,Paid Ads,Paid Ads_C,iOS,Standard
1,2,2024-04-01,2024-05-01,Male,Single,54,Premium,Netherlands,Referral,Referral_B,iOS,Standard
2,3,2024-05-01,2024-07-01,Male,Single,34,Medium,Poland,Paid Ads,Paid Ads_A,Android,Standard
3,4,2024-07-01,NaN,Male,Married,38,High,Belgium,Organic,Organic_C,Android,Standard
4,5,2024-03-01,2024-04-01,Male,Single,25,Low,Sweden,Paid Ads,Paid Ads_A,Android,Basic


In [4]:
df_cohort.shape

(12000, 12)

#### Exploratory Questions | Who are our customers?

In [5]:
# Missing Values
df_cohort.isna().sum()

user_id                  0
acquisition_date         0
cancellation_month    4934
gender                   0
marital_status           0
age                      0
income_segment         631
country                  0
channel                  0
campaign_id              0
device_type              0
plan_type                0
dtype: int64

In [6]:
# Numerical Summary
df_cohort.describe()

,user_id,age
count,12000.00000,12000.000000
mean,6000.50000,34.630250
std,3464.24595,9.514896
min,1.00000,18.000000
25%,3000.75000,28.000000
50%,6000.50000,34.000000
75%,9000.25000,41.000000
max,12000.00000,65.000000


In [7]:
categorical_cols = [
    "gender",
    "marital_status",
    "income_segment",
    "country",
    "channel",
    "campaign_id",
    "device_type",
    "plan_type"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df_cohort[col].value_counts())


--- gender ---
gender
Female    6016
Male      5984
Name: count, dtype: int64

--- marital_status ---
marital_status
Single     7087
Married    4913
Name: count, dtype: int64

--- income_segment ---
income_segment
High       4823
Medium     4267
Low        1627
Premium     652
Name: count, dtype: int64

--- country ---
country
Netherlands       1057
Austria           1031
Poland            1029
Italy             1028
Belgium           1016
Sweden             995
Switzerland        992
Czech Republic     980
Portugal           972
France             969
Spain              968
Germany            963
Name: count, dtype: int64

--- channel ---
channel
Paid Ads    4854
Organic     4690
Referral    2456
Name: count, dtype: int64

--- campaign_id ---
campaign_id
Paid Ads_A    1636
Organic_B     1616
Paid Ads_B    1612
Paid Ads_C    1606
Organic_A     1552
Organic_C     1522
Referral_A     830
Referral_C     816
Referral_B     810
Name: count, dtype: int64

--- device_type ---
device_type
And

In [8]:
# Acquisition Trend
acquisition_trend = (
    df_cohort.groupby("acquisition_date", as_index=False)
    .agg(customers=("user_id", "count"))
)

fig = px.line(
    acquisition_trend,
    x="acquisition_date",
    y="customers",
    title="Customer Acquisition Trend"
)

fig.show()

In [9]:
df_cohort["acquisition_date"] = pd.to_datetime(df_cohort["acquisition_date"])

In [10]:
df_cohort["acquisition_month"] = df_cohort["acquisition_date"].dt.to_period("M").astype(str)
df_cohort.head()

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type,acquisition_month
0,1,2024-07-01,2025-02-01,Male,Married,31,Medium,Germany,Paid Ads,Paid Ads_C,iOS,Standard,2024-07
1,2,2024-04-01,2024-05-01,Male,Single,54,Premium,Netherlands,Referral,Referral_B,iOS,Standard,2024-04
2,3,2024-05-01,2024-07-01,Male,Single,34,Medium,Poland,Paid Ads,Paid Ads_A,Android,Standard,2024-05
3,4,2024-07-01,NaN,Male,Married,38,High,Belgium,Organic,Organic_C,Android,Standard,2024-07
4,5,2024-03-01,2024-04-01,Male,Single,25,Low,Sweden,Paid Ads,Paid Ads_A,Android,Basic,2024-03


In [11]:
acquisition_trend = (
    df_cohort.groupby("acquisition_month", as_index=False)
    .agg(customers=("user_id", "count"))
)

fig = px.bar(
    acquisition_trend,
    x="acquisition_month",
    y="customers",
    title="Monthly Acquisition Trend"
)

fig.show()

In [12]:
# Identify peak month
top_month = acquisition_trend.loc[
    acquisition_trend["customers"].idxmax(), "acquisition_month"
]

top_month

'2024-05'

#### Creating Highlight column to mention

In [13]:
# Highlight column
acquisition_trend["highlight"] = np.where(
    acquisition_trend["acquisition_month"] == top_month,
    "Peak",
    "Other"
)

In [14]:
# Now, we can build more informative plot.
fig = px.bar(
    acquisition_trend,
    x="acquisition_month",
    y="customers",
    color = 'highlight',
    title="Monthly Acquisition Trend"
)

fig.show()

#### Gender Distribution

In [15]:
gender_counts = df_cohort["gender"].value_counts().reset_index()
gender_counts.columns = ["gender", "count"]

fig = px.bar(
    gender_counts,
    x="gender",
    y="count",
    title="Gender Distribution"
)

fig.show()

#### Marital Status Distribution

In [16]:
marital_counts = df_cohort["marital_status"].value_counts().reset_index()
marital_counts.columns = ["marital_status", "count"]

fig = px.bar(
    marital_counts,
    x="marital_status",
    y="count",
    title="Marital Status Distribution"
)

fig.show()

#### Age Distribution

In [17]:
fig = px.histogram(
    df_cohort,
    x="age",
    nbins=30,
    title="Age Distribution"
)

fig.show()

#### Country Distribution

In [18]:
country_counts = df_cohort["country"].value_counts().reset_index()
country_counts.columns = ["country", "count"]

fig = px.bar(
    country_counts,
    x="country",
    y="count",
    title="Country Distribution"
)

fig.show()

#### Channel Distribution

In [19]:
channel_counts = df_cohort["channel"].value_counts().reset_index()
channel_counts.columns = ["channel", "count"]

fig = px.bar(
    channel_counts,
    x="channel",
    y="count",
    title="Channel Distribution"
)

fig.show()

#### Acquisition Month vs Gender

In [20]:
df_cohort["acquisition_month"] = df_cohort["acquisition_date"].dt.to_period("M").dt.to_timestamp()

gender_trend = (
    df_cohort.groupby(["acquisition_month", "gender"], as_index=False)
    .agg(count=("user_id", "count"))
)

fig = px.bar(
    gender_trend,
    x="acquisition_month",
    y="count",
    color="gender",
    barmode="group",
    title="Acquisition by Gender Over Time"
)

fig.show()

#### Acquisition Month vs Marital Status

In [21]:
marital_trend = (
    df_cohort.groupby(["acquisition_month", "marital_status"], as_index=False)
    .agg(count=("user_id", "count"))
)

fig = px.bar(
    marital_trend,
    x="acquisition_month",
    y="count",
    color="marital_status",
    barmode="group",
    title="Acquisition by Marital Status Over Time"
)

fig.show()

#### Acquisition Month vs Country

In [22]:
country_trend = (
    df_cohort.groupby(["acquisition_month", "country"], as_index=False)
    .agg(count=("user_id", "count"))
)

fig = px.bar(
    country_trend,
    x="acquisition_month",
    y="count",
    color="country",
    barmode="group",
    title="Acquisition by Country Over Time"
)

fig.show()

##### Country-Region Mapper

In [23]:
region_map = {
    "France": "Western Europe",
    "Germany": "Western Europe",
    "Netherlands": "Western Europe",
    "Belgium": "Western Europe",

    "Spain": "Southern Europe",
    "Italy": "Southern Europe",
    "Portugal": "Southern Europe",

    "Poland": "Eastern Europe",
    "Czech Republic": "Eastern Europe",
    "Hungary": "Eastern Europe",
    "Romania": "Eastern Europe",

    "Sweden": "Northern Europe",
    "Norway": "Northern Europe",
    "Denmark": "Northern Europe",
    "Finland": "Northern Europe"
}

#### Preparing the Data

In [24]:
df_cohort["region"] = df_cohort["country"].map(region_map)

#Step 2: Aggregate
country_trend = (
    df_cohort.groupby(["acquisition_month", "region", "country"], as_index=False)
    .agg(count=("user_id", "count"))
)

#### Visualizing the results | Subplots

In [25]:
fig = px.bar(
    country_trend,
    x="acquisition_month",
    y="count",
    color="country",
    # barmode="group",
    facet_col="region",
    facet_col_wrap=2,
    title="Acquisition by Country (Split by Region)"
)

fig.show()

In [26]:
fig = px.bar(
    country_trend,
    x="acquisition_month",
    y="count",
    color="country",
    facet_col="region",
    facet_col_wrap=2,
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="Acquisition by Country | 1"
)

fig.show()

In [27]:
fig = px.bar(
    country_trend,
    x="acquisition_month",
    y="count",
    color="country",
    facet_col="region",
    facet_col_wrap=2,
    color_discrete_sequence =  px.colors.qualitative.Plotly,
    title="Acquisition by Country | 2 "
)

fig.show()

In [28]:
fig = px.bar(
    country_trend,
    x="acquisition_month",
    y="count",
    color="country",
    facet_col="region",
    facet_col_wrap=2,
    color_discrete_sequence = px.colors.qualitative.Dark24,
    title="Acquisition by Country | 3"
)

fig.show()

In [29]:
fig = px.bar(
    country_trend,
    x="acquisition_month",
    y="count",
    color="country",
    facet_col="region",
    facet_col_wrap=2,
    color_discrete_sequence=px.colors.qualitative.Safe,
    title="Acquisition by Country (Styled Colors)"
)

fig.show()

#### Custom Color Pallet

In [30]:
color_map = {
    "France": "#1f77b4",
    "Germany": "#ff7f0e",
    "Netherlands": "#2ca02c",
    "Belgium": "#d62728",
    "Spain": "#9467bd",
    "Italy": "#8c564b",
    "Portugal": "#e377c2",
    "Poland": "#7f7f7f",
    "Czech Republic": "#bcbd22",
    "Sweden": "#17becf"
}

#### Acquisition Month vs Channel

In [31]:
channel_trend = (
    df_cohort.groupby(["acquisition_month", "channel"], as_index=False)
    .agg(count=("user_id", "count"))
)

fig = px.bar(
    channel_trend,
    x="acquisition_month",
    y="count",
    color="channel",
    barmode="group",
    title="Acquisition by Channel Over Time"
)

fig.show()

#### Acquisition Month vs Plan Type

In [32]:
plan_trend = (
    df_cohort.groupby(["acquisition_month", "plan_type"], as_index=False)
    .agg(count=("user_id", "count"))
)

fig = px.bar(
    plan_trend,
    x="acquisition_month",
    y="count",
    color="plan_type",
    barmode="group",
    title="Acquisition by Plan Type Over Time"
)

fig.show()

#### Cohort Analysis

In [33]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px

In [34]:
df = pd.read_csv('../data/cohort/cohort_analysis.csv',
                 parse_dates = ["acquisition_date", 'cancellation_month'])
df.head()

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type
0,1,2024-07-01,2025-02-01,Male,Married,31,Medium,Germany,Paid Ads,Paid Ads_C,iOS,Standard
1,2,2024-04-01,2024-05-01,Male,Single,54,Premium,Netherlands,Referral,Referral_B,iOS,Standard
2,3,2024-05-01,2024-07-01,Male,Single,34,Medium,Poland,Paid Ads,Paid Ads_A,Android,Standard
3,4,2024-07-01,NaT,Male,Married,38,High,Belgium,Organic,Organic_C,Android,Standard
4,5,2024-03-01,2024-04-01,Male,Single,25,Low,Sweden,Paid Ads,Paid Ads_A,Android,Basic


In [35]:
df.shape

(12000, 12)

In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   user_id             12000 non-null  int64         
 1   acquisition_date    12000 non-null  datetime64[us]
 2   cancellation_month  7066 non-null   datetime64[us]
 3   gender              12000 non-null  str           
 4   marital_status      12000 non-null  str           
 5   age                 12000 non-null  int64         
 6   income_segment      11369 non-null  str           
 7   country             12000 non-null  str           
 8   channel             12000 non-null  str           
 9   campaign_id         12000 non-null  str           
 10  device_type         12000 non-null  str           
 11  plan_type           12000 non-null  str           
dtypes: datetime64[us](2), int64(2), str(8)
memory usage: 1.7 MB


In [37]:
df.isnull().sum()

user_id                  0
acquisition_date         0
cancellation_month    4934
gender                   0
marital_status           0
age                      0
income_segment         631
country                  0
channel                  0
campaign_id              0
device_type              0
plan_type                0
dtype: int64

In [38]:
df.describe(include="all")

,user_id,acquisition_date,cancellation_month,gender,marital_status,age,income_segment,country,channel,campaign_id,device_type,plan_type
count,12000.00000,12000,7066,12000,12000,12000.000000,11369,12000,12000,12000,12000,12000
unique,NaN,NaN,NaN,2,2,NaN,4,12,3,9,3,3
top,NaN,NaN,NaN,Female,Single,NaN,High,Netherlands,Paid Ads,Paid Ads_A,Android,Standard
freq,NaN,NaN,NaN,6016,7087,NaN,4823,1057,4854,1636,6582,5979
mean,6000.50000,2024-04-15 20:46:48,2024-07-28 15:31:07.761109,NaN,NaN,34.630250,NaN,NaN,NaN,NaN,NaN,NaN
min,1.00000,2024-01-01 00:00:00,2024-02-01 00:00:00,NaN,NaN,18.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,3000.75000,2024-02-01 00:00:00,2024-05-01 00:00:00,NaN,NaN,28.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,6000.50000,2024-04-01 00:00:00,2024-07-01 00:00:00,NaN,NaN,34.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,9000.25000,2024-06-01 00:00:00,2024-10-01 00:00:00,NaN,NaN,41.000000,NaN,NaN,NaN,NaN,NaN,NaN
max,12000.00000,2024-08-01 00:00:00,2025-08-01 00:00:00,NaN,NaN,65.000000,NaN,NaN,NaN,NaN,NaN,NaN


#### Data Preperation | Cohort Month

In [39]:
df["acquisition_month"] = df["acquisition_date"].dt.to_period("M")

df[["user_id", "acquisition_date", "acquisition_month"]].head()

,user_id,acquisition_date,acquisition_month
0,1,2024-07-01,2024-07
1,2,2024-04-01,2024-04
2,3,2024-05-01,2024-05
3,4,2024-07-01,2024-07
4,5,2024-03-01,2024-03


In [40]:
df["is_churned"] = df["cancellation_month"].notna().astype(int)
df["is_churned"].value_counts()

is_churned
1    7066
0    4934
Name: count, dtype: int64

In [41]:
df["acquisition_date"] = pd.to_datetime(df["acquisition_date"], errors="coerce")
df["cancellation_month"] = pd.to_datetime(df["cancellation_month"], errors="coerce")

df["month_churned"] = np.where(
    df["cancellation_month"].notna(),
    (
        (df["cancellation_month"].dt.year - df["acquisition_date"].dt.year) * 12
        + (df["cancellation_month"].dt.month - df["acquisition_date"].dt.month)
    ),
    np.nan
)

df[["user_id", "acquisition_date", "acquisition_month", "cancellation_month", "month_churned"]].head(10)

,user_id,acquisition_date,acquisition_month,cancellation_month,month_churned
0,1,2024-07-01,2024-07,2025-02-01,7.0
1,2,2024-04-01,2024-04,2024-05-01,1.0
2,3,2024-05-01,2024-05,2024-07-01,2.0
3,4,2024-07-01,2024-07,NaT,NaN
4,5,2024-03-01,2024-03,2024-04-01,1.0
5,6,2024-08-01,2024-08,NaT,NaN
6,7,2024-05-01,2024-05,NaT,NaN
7,8,2024-05-01,2024-05,2024-06-01,1.0
8,9,2024-07-01,2024-07,NaT,NaN
9,10,2024-02-01,2024-02,2024-12-01,10.0


In [42]:
df["month_churned"] = df["month_churned"].where(
    df["month_churned"].between(1, 12),
    np.nan
)

df["month_churned"].value_counts(dropna=False).sort_index()

month_churned
1.0     2392
2.0     1363
3.0      845
4.0      585
5.0      443
6.0      362
7.0      266
8.0      248
9.0      182
10.0     168
11.0     113
12.0      99
NaN     4934
Name: count, dtype: int64

####  Cohort Size

In [43]:
cohort_size = (
    df.groupby("acquisition_month")
    .agg(new_users=("user_id", "count"))
    .reset_index()
)

cohort_size

,acquisition_month,new_users
0,2024-01,1502
1,2024-02,1528
2,2024-03,1465
3,2024-04,1509
4,2024-05,1541
5,2024-06,1493
6,2024-07,1500
7,2024-08,1462


In [44]:
cohort_churn = (
    df[df["month_churned"].notna()]
    .groupby(["acquisition_month", "month_churned"])
    .agg(users=("user_id", "count"))
    .reset_index()
)

cohort_churn["month_churned"] = cohort_churn["month_churned"].astype(int)

cohort_churn.head(15)

,acquisition_month,month_churned,users
0,2024-01,1,148
1,2024-01,2,95
2,2024-01,3,79
3,2024-01,4,76
4,2024-01,5,52
5,2024-01,6,49
6,2024-01,7,41
7,2024-01,8,37
8,2024-01,9,24
9,2024-01,10,26


In [45]:
max_month = 12
tenure_range = np.arange(1, max_month + 1)

full_index = pd.MultiIndex.from_product(
    [cohort_size["acquisition_month"].unique(), tenure_range],
    names=["acquisition_month", "month_churned"]
)

cohort_data = (
    cohort_churn
    .set_index(["acquisition_month",  "month_churned"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

cohort_data.head(15)

,acquisition_month,month_churned,users
0,2024-01,1,148
1,2024-01,2,95
2,2024-01,3,79
3,2024-01,4,76
4,2024-01,5,52
5,2024-01,6,49
6,2024-01,7,41
7,2024-01,8,37
8,2024-01,9,24
9,2024-01,10,26


#### Merge Cohort Size

In [46]:
cohort_data = cohort_data.merge(cohort_size, on="acquisition_month", how="left")
cohort_data.head(15)

,acquisition_month,month_churned,users,new_users
0,2024-01,1,148,1502
1,2024-01,2,95,1502
2,2024-01,3,79,1502
3,2024-01,4,76,1502
4,2024-01,5,52,1502
5,2024-01,6,49,1502
6,2024-01,7,41,1502
7,2024-01,8,37,1502
8,2024-01,9,24,1502
9,2024-01,10,26,1502


####  Calculate Metrics

In [48]:
cohort_data = cohort_data.sort_values(["acquisition_month", "month_churned"])

cohort_data["cumulative_churn"] = (
    cohort_data.groupby("acquisition_month")["users"].cumsum()
)

cohort_data["active_users"] = (
    cohort_data["new_users"] - cohort_data["cumulative_churn"]
)

cohort_data["retention_rate"] = (
    cohort_data["active_users"] / cohort_data["new_users"]
)

cohort_data["acquisition_month"] = cohort_data["acquisition_month"].astype(str)

cohort_data.head(15)

,acquisition_month,month_churned,users,new_users,cumulative_churn,active_users,retention_rate
0,2024-01,1,148,1502,148,1354,0.901465
1,2024-01,2,95,1502,243,1259,0.838216
2,2024-01,3,79,1502,322,1180,0.785619
3,2024-01,4,76,1502,398,1104,0.735020
4,2024-01,5,52,1502,450,1052,0.700399
5,2024-01,6,49,1502,499,1003,0.667776
6,2024-01,7,41,1502,540,962,0.640479
7,2024-01,8,37,1502,577,925,0.615846
8,2024-01,9,24,1502,601,901,0.599867
9,2024-01,10,26,1502,627,875,0.582557


#### Cohort Retention Heatmap

In [49]:
heatmap_data = cohort_data.pivot(
    index="acquisition_month",
    columns="month_churned",
    values="retention_rate"
)

heatmap_data

month_churned,1,2,3,4,5,6,7,8,9,10,11,12
acquisition_month,,,,,,,,,,,,
2024-01,0.901465,0.838216,0.785619,0.735020,0.700399,0.667776,0.640479,0.615846,0.599867,0.582557,0.575233,0.566578
2024-02,0.898560,0.826571,0.767670,0.726440,0.685864,0.649869,0.628272,0.607330,0.587042,0.569372,0.553010,0.540576
2024-03,0.651195,0.494198,0.423208,0.379522,0.345392,0.325597,0.305802,0.291468,0.283959,0.273038,0.270307,0.264164
2024-04,0.644798,0.487078,0.408880,0.370444,0.341948,0.322068,0.308151,0.298211,0.287608,0.280318,0.273691,0.269052
2024-05,0.755354,0.603504,0.522388,0.467229,0.420506,0.384166,0.363400,0.345230,0.328358,0.315380,0.305646,0.299805
2024-06,0.744139,0.582050,0.473543,0.409243,0.373074,0.344943,0.318151,0.299397,0.287341,0.270596,0.260549,0.252512
2024-07,0.905333,0.823333,0.773333,0.722000,0.688000,0.651333,0.627333,0.596000,0.580000,0.561333,0.550667,0.539333
2024-08,0.903557,0.841313,0.778386,0.733242,0.692886,0.661423,0.638167,0.610807,0.588919,0.578659,0.567031,0.558140


In [ ]:
fig = px.imshow(
    heatmap_data,
    aspect="auto",
    color_continuous_scale="Blues",
    text_auto=".1%",
    labels={
        "x": "Months Since Acquisition",
        "y": "Acquisition Month",
        "color": "Retention Rate"
    },
    title="Cohort Retention Heatmap"
)

fig.update_layout(
    xaxis_title="Months Since Acquisition",
    yaxis_title="Acquisition Month"
)

fig.show()

: 